## 1) Setup and Imports

In [ ]:
import os
import sys
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal, norm

src_path = os.path.join("..", "src")
abs_src_path = os.path.abspath(src_path)
if abs_src_path not in sys.path:
    sys.path.insert(0, abs_src_path)

from DVC.objects import vine_obj_bin, margin_obj
from DVC.preparation import prep_cop
from DVC.vine_model import fit_vine, evaluate_vine, sample_vine
from DVC.info_estimation import vine_entropy

%matplotlib inline

print("Imports successful!")

## 2) Generate Synthetic Data

In [ ]:
# np.random.seed(42)
# n_samples = 3000
# dim = 3
# # We'll create random data in [0,1]^3
# x = np.random.rand(n_samples, dim)

# print("Data shape:", x.shape)

# In[2]: Generate correlated data in R^5

np.random.seed(123)
dim = 5
n_samples = 3000

# Example Covariance matrix (must be SPD). Here is a small arbitrary choice:
cov_mat = np.array([
    [1.0, 0.5, 0.3, 0.1, 0.2],
    [0.5, 1.0, 0.4, 0.2, 0.1],
    [0.3, 0.4, 1.0, 0.6, 0.0],
    [0.1, 0.2, 0.6, 1.0, 0.5],
    [0.2, 0.1, 0.0, 0.5, 1.0]
])
mean_vec = np.zeros(dim)

# Sample
x = multivariate_normal(mean=mean_vec, cov=cov_mat).rvs(size=n_samples)
print("X shape:", x.shape)

# Convert X_real to [0,1]^dim by taking each column's Normal CDF
# i.e. for each column x_i, we do u_i = Phi(x_i) => shape [n_samples, dim]
X_unif = norm.cdf(x)
print("X_unif shape:", X_unif.shape)
print("First 5 rows:\n", X_unif[:5])


## 3) Build Margin Objects and Create the Vine

In [ ]:
# 1) Build margin objects (e.g. standard normal placeholders)
margin_vine = []
for i in range(dim):
    margin_vine.append(margin_obj(dist='norm', theta=[0.,1.], is_cont=True))

# 2) Create a vine_obj_bin
#    families='kercop' => nonparam approach
vine = vine_obj_bin(
    vine_family='c-vine',  # or 'c-vine','d-vine'
    families='kercop',
    vine_depth=dim,
    margin=margin_vine,
    knots=50,
    method='matrix',
    r_matrix=None
)

# 3) Create dictionary configs for fitting
gen_dict = {
    'parallel': True,
    'binning': False,
    'param': False,     # if True => param approach
    'vine_depth': dim,
    'fitted': False
}
npc_dict = {
    'opt_method': 'LL1',
    'batch_paral': 3
}
par_dict = {
    'param_families': ["ind","gaussian","student","clayton","claytonrot90"]
}
bin_dict = {
    'n_bin': 3
}

## 4) Fit the Vine

In [ ]:
# Fit the vine model using the above settings
vine.fit(x, gen_dict, npc_dict, par_dict, bin_dict)

print("Vine fitted with dimension =", vine.n_cop)
print("Number of copula objects stored:", len(vine.copulas))

## 5) Evaluate the Vine PDF at New Points

In [ ]:
# Generate random test points in [0,1]^3
test_pts = torch.rand((1000, dim), dtype=torch.float32)
p, p_cop, logmarg = vine.evaluation(test_pts)

print("Evaluated p shape:", p.shape)  # should be [1000]
print("First 5 p values:\n", p[:5])

## 6) Sample from the Vine

In [ ]:
samples_vine = vine.sample(500)
print("Samples from vine shape:", samples_vine.shape)
print("First 5 vine samples:\n", samples_vine[:5])

## 7) Approximate the Vine's Entropy

In [ ]:
info_dict = {
    'alpha': 0.05,
    'cases': 500,
    'iterations': 5
}
H_est = vine_entropy(vine, info_dict)
print("Approx. entropy from vine:", H_est)

## 8) Visualize Real Data vs. Vine Samples

In [ ]:
# We'll do a couple of scatter plots in 2D, for dimension 0 vs dimension 1
fig, axes = plt.subplots(1, 2, figsize=(10,5))

# Left: Original data, dimension 0 vs. dimension 1
axes[0].scatter(x[:,0], x[:,1], alpha=0.3)
axes[0].set_title("Real Data (dim 0 vs 1)")
axes[0].set_xlabel("x[:,0]")
axes[0].set_ylabel("x[:,1]")

# Right: Vine samples, dimension 0 vs dimension 1
axes[1].scatter(samples_vine[:,0], samples_vine[:,1], alpha=0.3, color='orange')
axes[1].set_title("Vine Samples (dim 0 vs 1)")
axes[1].set_xlabel("samples_vine[:,0]")
axes[1].set_ylabel("samples_vine[:,1]")

plt.tight_layout()
plt.show()

## 9) Compare Correlation

In [ ]:
# Compare correlation of real data vs vine-sampled data
corr_real = np.corrcoef(x, rowvar=False)
corr_vine = np.corrcoef(samples_vine, rowvar=False)

print("Correlation matrix of real data:\n", corr_real)
print("\nCorrelation matrix of vine-sampled data:\n", corr_vine)

# Plot correlation heatmaps (optional)
fig, ax = plt.subplots(1,2, figsize=(8,4))

cax1 = ax[0].imshow(corr_real, vmin=-1, vmax=1, cmap='coolwarm')
ax[0].set_title("Real Data Corr")
fig.colorbar(cax1, ax=ax[0], fraction=0.046, pad=0.04)

cax2 = ax[1].imshow(corr_vine, vmin=-1, vmax=1, cmap='coolwarm')
ax[1].set_title("Vine Sample Corr")
fig.colorbar(cax2, ax=ax[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()